# 03 — Joint nanoGPT-style Route Generation

## From Understanding to Generation

Notebook 02 used a **transformer encoder** (BERT-style) to *understand* routes and predict their grade. This notebook uses a **transformer decoder** (GPT-style) to *generate* new routes.

### The key difference: Encoder vs Decoder

| Aspect | BERT-style (Encoder) | GPT-style (Decoder) |
|---|---|---|
| Attention | Bidirectional (sees all tokens) | Causal (only sees past tokens) |
| Training | Masked language modeling | Next-token prediction |
| Use case | Classification, regression | Text generation |
| Output | Single prediction per sequence | One prediction per position |

### How GPT-style generation works

The model is trained to predict the **next token** given all previous tokens:

```text
Input:  <BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>
Target: <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start>
```

At generation time, we:
1. Start with a prompt like `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>`
2. Ask the model to predict the next token
3. Sample from the predicted probability distribution
4. Append the sampled token to the sequence
5. Repeat until we generate `<EOS>` or hit a max length

### Conditioning on board, angle, and grade

The prompt tokens tell the model *what kind of route to generate*:
- `<BOARD_TB2>`: Generate a route for the Tension Board 2
- `<ANGLE_40>`: At 40 degrees
- `<GRADE_V6>`: At V6 difficulty

This is analogous to how ChatGPT uses a system prompt to condition its responses.

In [ ]:
from pathlib import Path
import sys
import json
import math
import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from climbingboardgpt.config import load_board_configs
from climbingboardgpt.datasets import RouteGPTDataset
from climbingboardgpt.generation import generate_one
from climbingboardgpt.models import JointRouteGPT

In [ ]:
TOKENIZED = ROOT / "data" / "processed" / "tokenized"
df_routes = pd.read_csv(TOKENIZED / "route_sequences.csv")
vocab = json.loads((TOKENIZED / "token_vocab.json").read_text(encoding="utf-8"))
stoi = {str(k): int(v) for k, v in vocab["stoi"].items()}
itos = {int(k): str(v) for k, v in vocab["itos"].items()}

pad_id = stoi["<PAD>"]
unk_id = stoi["<UNK>"]

print(f"Vocabulary size: {len(stoi):,}")
print(f"Total routes: {len(df_routes):,}")

## Sequence encoding for causal language modeling

### The autoregressive setup

For GPT-style training, each route becomes a sequence where the model learns to predict each token given all previous tokens:

```text
Input:   <BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> <TB2_p369_middle>
Target:  <BOARD_TB2> <ANGLE_40> <GRADE_V6> <TB2_p344_start> <TB2_p369_middle> <TB2_p603_finish>
```

The input is shifted right by one position compared to the target. This is the standard causal language modeling setup.

### Why include the grade in the training sequence?

For the grade predictor (notebook 02), we excluded the grade because the model needed to predict it. But for the generator, we **include** the grade (`<GRADE_V6>`) in the training data so the model learns the relationship between grade and hold selection.

At generation time, we provide the grade as part of the prompt, and the model generates holds that are appropriate for that grade.

In [ ]:
def encode(tokens):
    """Convert token strings to integer IDs."""
    return [stoi.get(token, unk_id) for token in tokens]

# Use the "with grade" version for GPT training
# The model needs to see the grade to learn grade-hold relationships
df_routes["gpt_tokens"] = df_routes["sequence_with_grade"].fillna("").str.split()
df_routes["gpt_ids"] = df_routes["gpt_tokens"].apply(encode)
df_routes["seq_len"] = df_routes["gpt_ids"].apply(len)
max_len = int(df_routes["seq_len"].max())
block_size = max_len - 1  # Input length (one less than full sequence)

# Create train/val splits
train_df = df_routes[df_routes["split"] == "train"].reset_index(drop=True)
val_df = df_routes[df_routes["split"] == "val"].reset_index(drop=True)

# Create datasets and data loaders
# RouteGPTDataset handles the input/target shift for causal modeling
train_ds = RouteGPTDataset(train_df, max_len=max_len, pad_id=pad_id)
val_ds = RouteGPTDataset(val_df, max_len=max_len, pad_id=pad_id)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

print(f"Max sequence length: {max_len}")
print(f"Block size (input length): {block_size}")
print(f"Training samples: {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")

## The GPT Model Architecture

### JointRouteGPT

This is a **causal transformer decoder** — the same architecture used in GPT-2, GPT-3, etc., but much smaller:

1. **Token embeddings**: Convert integer token IDs to dense vectors
2. **Positional embeddings**: Learned position vectors (not sinusoidal)
3. **Causal self-attention**: Each position can only attend to previous positions (via a causal mask)
4. **Transformer layers**: Multiple layers of attention + feedforward
5. **Language modeling head**: Projects hidden states to vocabulary logits

### Key hyperparameters

- `n_embd=128`: Embedding dimension (GPT-2 small uses 768)
- `n_head=4`: Number of attention heads
- `n_layer=4`: Number of transformer layers (GPT-2 small uses 12)
- `dropout=0.10`: Dropout probability

This is intentionally small — we're training on ~40K short sequences, not billions of long documents.

### Weight tying

The output projection layer shares weights with the token embedding layer (`self.lm_head.weight = self.token_emb.weight`). This is a common technique that:
- Reduces parameter count
- Acts as a regularizer
- Is used in GPT-2 and many other language models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = JointRouteGPT(
    vocab_size=len(stoi),
    block_size=block_size,
    n_embd=128,
    n_head=4,
    n_layer=4,
    dropout=0.10,
    pad_id=pad_id,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)

print(f"Device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def train_epoch():
    """Train for one epoch."""
    model.train()
    losses = []
    n = 0
    for batch in train_loader:
        x = batch["input_ids"].to(device)
        y = batch["target_ids"].to(device)
        
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(x, y)
        loss.backward()
        
        # Gradient clipping prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        losses.append(loss.item() * x.size(0))
        n += x.size(0)
    return sum(losses) / max(1, n)

@torch.no_grad()
def eval_loss(loader):
    """Evaluate loss on a data loader."""
    model.eval()
    losses = []
    n = 0
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["target_ids"].to(device)
        _, loss = model(x, y)
        losses.append(loss.item() * x.size(0))
        n += x.size(0)
    return sum(losses) / max(1, n)

## Training

### What we're optimizing

The model minimizes **cross-entropy loss** — the standard loss function for language modeling. At each position, the model outputs a probability distribution over the entire vocabulary, and the loss measures how surprised it is by the actual next token.

### Perplexity

We also track **perplexity**, which is `exp(loss)`. Perplexity answers the question: "On average, how many tokens was the model choosing between at each step?" Lower perplexity = better model.

For reference:
- A model that always predicts the right token has perplexity = 1
- A model that picks uniformly from a 1000-token vocab has perplexity = 1000
- Good language models on English text achieve perplexity ~15-20

Our vocabulary is ~4000+ tokens, so a perplexity significantly below that indicates the model is learning meaningful patterns.

In [ ]:
history = []
best_val_loss = float("inf")
best_state = None
patience = 10
stagnant = 0

print("Starting GPT training...\n")

for epoch in range(1, 21):
    train_loss = train_epoch()
    val_loss = eval_loss(val_loader)
    
    # Track perplexity (exponentiated loss)
    train_ppl = math.exp(min(train_loss, 20))
    val_ppl = math.exp(min(val_loss, 20))
    
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_perplexity": train_ppl,
        "val_perplexity": val_ppl,
    })
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stagnant = 0
    else:
        stagnant += 1
    
    if epoch == 1 or epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | "
              f"Val PPL: {val_ppl:.1f}")
    
    if stagnant >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Load best model
if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest validation loss: {best_val_loss:.4f}")
print(f"Best validation perplexity: {math.exp(min(best_val_loss, 20)):.1f}")

## Generating Routes

### The generation process

To generate a route, we:

1. **Create a prompt**: `<BOS> <BOARD_TB2> <ANGLE_40> <GRADE_V6>`
2. **Feed it to the model**: Get a probability distribution over the vocabulary for the next token
3. **Sample a token**: Use temperature and top-k filtering to control randomness
4. **Append and repeat**: Add the sampled token to the sequence and repeat until `<EOS>` or max length

### Temperature and top-k sampling

- **Temperature** (default 0.9): Controls randomness. Lower = more deterministic, higher = more random
- **Top-k** (default 50): Only consider the k most likely tokens. This prevents the model from generating very unlikely tokens.

These are the same techniques used in language models like GPT-3 to control output diversity.

In [ ]:
# Generate sample routes for both boards
configs = load_board_configs(["tb2", "kilter"])
configs_by_key = {config.board_key: config for config in configs}

samples = []
for board_key, config in configs_by_key.items():
    for grouped_v in [3, 5, 7]:  # V3, V5, V7
        sample = generate_one(
            model=model,
            stoi=stoi,
            itos=itos,
            device=device,
            board_prefix=config.token_prefix,
            angle=40,
            grouped_v=grouped_v,
            role_name_to_id=config.role_definitions,
            temperature=0.9,
            top_k=50,
            max_new_tokens=40,
        )
        samples.append({"board_key": board_key, **sample})

samples_df = pd.DataFrame(samples)
print("Generated route samples:")
print(samples_df[["board_key", "requested_grouped_v", "basic_valid", "sequence", "frames"]])

## Generate More Routes for Evaluation

Notebook 04 needs a larger set of generated routes for meaningful evaluation. Let's generate routes across multiple angles and grades for both boards.

In [ ]:
# Generate routes across multiple angles and grades for evaluation
all_samples = []

for board_key, config in configs_by_key.items():
    # Get common angles and grades for this board
    board_df = df_routes[df_routes["board_key"] == board_key]
    common_angles = sorted(board_df["angle"].astype(int).value_counts().head(5).index.tolist())
    common_grades = sorted(board_df["grouped_v"].astype(int).value_counts().head(8).index.tolist())
    
    print(f"\nGenerating for {config.display_name}:")
    print(f"  Angles: {common_angles}")
    print(f"  Grades: V{min(common_grades)}-V{max(common_grades)}")
    
    for angle in common_angles:
        for grade in common_grades:
            for i in range(5):  # 5 samples per condition
                sample = generate_one(
                    model=model,
                    stoi=stoi,
                    itos=itos,
                    device=device,
                    board_prefix=config.token_prefix,
                    angle=int(angle),
                    grouped_v=int(grade),
                    role_name_to_id=config.role_definitions,
                    temperature=0.9,
                    top_k=50,
                    max_new_tokens=40,
                )
                all_samples.append({"board_key": board_key, **sample})

all_samples_df = pd.DataFrame(all_samples)
print(f"\nTotal generated routes: {len(all_samples_df):,}")
print("\nBasic validity by board:")
print(all_samples_df.groupby("board_key")["basic_valid"].mean())

## Save Model and Generated Routes

We save the trained model checkpoint and generated routes for use in notebook 04 (evaluation).

In [ ]:
import os

# Save model checkpoint
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": {
        "vocab_size": len(stoi),
        "block_size": block_size,
        "n_embd": 128,
        "n_head": 4,
        "n_layer": 4,
        "dropout": 0.10,
        "pad_id": pad_id,
    },
    "stoi": stoi,
    "itos": {str(k): v for k, v in itos.items()},
    "best_val_loss": best_val_loss,
}
model_path = MODEL_DIR / "joint_route_gpt_generator.pth"
torch.save(checkpoint, model_path)
print(f"Saved model checkpoint to: {model_path}")

# Save training history
GEN_DIR = ROOT / "data" / "processed" / "generation"
GEN_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame(history).to_csv(GEN_DIR / "training_history.csv", index=False)
print(f"Saved training history to: {GEN_DIR / 'training_history.csv'}")

# Save generated routes (this is what notebook 04 needs)
all_samples_df.to_csv(GEN_DIR / "generated_routes.csv", index=False)
print(f"Saved {len(all_samples_df)} generated routes to: {GEN_DIR / 'generated_routes.csv'}")